# 01 — Getting Started with pytrist

This notebook walks through the basics of using `pytrist` to read Tristan-MP V2 simulation output:

1. Point `pytrist` at your output directory and inspect what is available
2. Read electromagnetic fields and plot a 2-D snapshot with field lines
3. Read particle data and make a phase-space scatter plot
4. Plot the energy history time series

**Before running:** set `SIM_DIR` in the next cell to the path of your simulation output directory.

In [ ]:
# ── USER CONFIGURATION ──────────────────────────────────────────────────────
SIM_DIR = "/path/to/your/simulation/output"   # <── edit this
STEP    = None   # None = last available step; or set e.g. STEP = 10
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import pytrist

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1  Inspect the simulation

In [ ]:
sim = pytrist.Simulation(SIM_DIR)

print("Available steps:", sim.steps)
print("Times [1/ωpe]: ", sim.times[:5], "...")
print()
print(sim.summary())

In [ ]:
# Resolve step
step = sim.steps[-1] if STEP is None else STEP
uc   = sim.unit_converter
p    = sim.params(step)

print(f"Using step {step}  |  t = {p.time:.1f} / ωpe = {uc.time(p.time):.3f} / Ωci")

## 2  Electromagnetic fields

In [ ]:
flds = sim.fields(step)

print("Available datasets:", flds.keys[:12], "...")
print("Bz shape:", flds.bz.shape)

In [ ]:
# For 3-D arrays, take the z=0 slice
def squeeze(arr):
    return arr[0] if arr.ndim == 3 else arr

bz = squeeze(flds.bz)
ny, nx = bz.shape

# Coordinate axes in ion inertial lengths
x = np.arange(nx) * uc.cell_to_di
y = np.arange(ny) * uc.cell_to_di

# Magnetic flux function for field-line contours
try:
    psi = squeeze(flds.psi())
except Exception:
    psi = None

fig, ax = plt.subplots(figsize=(9, 4))

vlim = np.percentile(np.abs(bz), 99)
im = ax.pcolormesh(
    x, y, bz,
    cmap="RdBu_r",
    norm=mcolors.TwoSlopeNorm(vmin=-vlim, vcenter=0, vmax=vlim),
    shading="auto",
)
fig.colorbar(im, ax=ax, label=r"$B_z / B_0$")

if psi is not None:
    ax.contour(x, y, psi, levels=20, colors="k", linewidths=0.4, alpha=0.5)

ax.set_xlabel(r"$x\;[d_i]$")
ax.set_ylabel(r"$y\;[d_i]$")
ax.set_aspect("equal")
ax.set_title(rf"$B_z$ at $t = {uc.time(p.time):.2f}\,\Omega_{{ci}}^{{-1}}$")
plt.tight_layout()
plt.show()

## 3  Particle data

> **Note:** Particle output is controlled by `prtl_enable` and `tot_output_stride` in the Tristan input file.
> If the cell below raises a `FileNotFoundError`, your run was configured with `prtl_enable = 0`.
> Set `prtl_enable = 1` in the input file and re-run the simulation to generate particle output.

In [ ]:
prtl = sim.particles(step)

# Species 1 = electrons, species 2 = ions (Tristan-V2 default)
electrons = prtl.species(1, units="ion")   # x, y in di; u, v, w in vAi
print(f"Number of electron macro-particles: {len(electrons['x']):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Phase space: x vs ux
ax = axes[0]
ax.scatter(electrons["x"], electrons["u"], s=0.2, alpha=0.3, rasterized=True)
ax.set_xlabel(r"$x\;[d_i]$")
ax.set_ylabel(r"$u_x\;[v_{Ai}]$")
ax.set_title("Electron phase space")

# Phase space: x vs uy
ax = axes[1]
ax.scatter(electrons["x"], electrons["v"], s=0.2, alpha=0.3, rasterized=True)
ax.set_xlabel(r"$x\;[d_i]$")
ax.set_ylabel(r"$u_y\;[v_{Ai}]$")
ax.set_title("Electron phase space")

plt.suptitle(rf"$t = {uc.time(p.time):.2f}\,\Omega_{{ci}}^{{-1}}$", y=1.01)
plt.tight_layout()
plt.show()

## 4  Energy history

In [ ]:
hist = sim.history()

print("History columns:", hist.column_names)

In [ ]:
t_ion = hist.time_ion   # time in 1/Ωci

fig, ax = plt.subplots(figsize=(8, 4))

if "totKinE" in hist.column_names:
    e_kin = hist["totKinE"]
    e_kin0 = e_kin[0] if e_kin[0] != 0 else 1.0
    ax.plot(t_ion, e_kin / e_kin0, label=r"$E_{\rm kin}$")

if "totEmE" in hist.column_names:
    e_em = hist["totEmE"]
    e_em0 = e_em[0] if e_em[0] != 0 else 1.0
    ax.plot(t_ion, e_em / e_em0, label=r"$E_{\rm EM}$")

ax.set_xlabel(r"$t\;[\Omega_{ci}^{-1}]$")
ax.set_ylabel("Energy / initial energy")
ax.set_title("Global energy history")
ax.legend()
plt.tight_layout()
plt.show()